# ProximityPrep — exploration

**Commerces** : compte par catégorie OSM, rayons 100→500 m (pas 100 m).

**Plage** : présence 0/1, rayons 1→5 km (pas 1 km).

Entrée = identité RodPrep (`hotel_code` Accor + `hotel_lat`/`hotel_lon`).

> Si un `ImportError` apparaît après une mise à jour du code : **Kernel → Restart & Run All**.

In [16]:
from pathlib import Path
import importlib
import sys

import pandas as pd

# --- Racine projet (contient prepare/ et rod_ia/) — indépendant du cwd du kernel ---
HERE = Path.cwd().resolve()
PROJECT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "prepare" / "proximity_prep").is_dir() and (candidate / "rod_ia").is_dir():
        PROJECT = candidate
        break
if PROJECT is None:
    raise RuntimeError(
        f"Racine projet introuvable depuis {HERE}. "
        "Ouvrir le notebook depuis le repo hotels/ ou définir le cwd du kernel."
    )

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Recharge le package (évite un kernel qui garde l'ancien __init__ en mémoire)
for mod_name in list(sys.modules):
    if mod_name == "prepare" or mod_name.startswith("prepare."):
        del sys.modules[mod_name]

from prepare.paths import default_paths
from prepare.proximity_prep.features import (
    BEACH_RADII_KM,
    COMMERCE_RADII_M,
    SHOP_CATEGORIES,
    ProximityFeatures,
    beach_presence_flags,
    count_commerce_by_category,
)
from prepare.proximity_prep.prep import ProximityPrep

paths = default_paths()
INPUT_DIR = paths.proximity_input
OUTPUT_DIR = paths.proximity_output
ROD_OUTPUT = paths.rod_output

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

print("PROJECT :", PROJECT)
print("Rayons commerces (m):", COMMERCE_RADII_M)
print("Rayons plage (km):", BEACH_RADII_KM)
print("Nb catégories shop:", len(SHOP_CATEGORIES))
print("Catégories:", SHOP_CATEGORIES)

PROJECT : /media/laghmari/ssd-data/dev/hotels
Rayons commerces (m): (100, 200, 300, 400, 500)
Rayons plage (km): (1, 2, 3, 4, 5)
Nb catégories shop: 15
Catégories: ('convenience', 'bakery', 'supermarket', 'alcohol', 'confectionery', 'beverages', 'grocery', 'ice_cream', 'fast_food', 'cosmetics', 'gift', 'tobacco', 'kiosk', 'pharmacy', 'chemist')


## 1. Entrée depuis RodPrep

In [17]:
prep = ProximityPrep(INPUT_DIR, OUTPUT_DIR)
prep.fill_input_from_rod(ROD_OUTPUT)
hotels = prep.load_input()
print(f"Hôtels : {len(hotels)} — codes : {hotels['hotel_code'].tolist()}")
hotels

Hôtels : 7 — codes : ['H2075', 'HB6A3', 'H0815', 'H6188', 'H0373', 'HB5I0', 'H3546']


,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_lat,hotel_lon
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,43.689186,7.240512
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,48.591522,7.754599
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,49.006733,2.519843
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,48.833827,2.256274
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,48.885048,2.329923
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,45.859165,6.619055
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,48.849778,2.282836


## 2. Calcul pur sur un point (`ProximityFeatures`)

Commerces 100–500 m + plage 1–5 km pour le premier hôtel (appels Overpass).

In [18]:
row0 = hotels.iloc[0]
lat, lon = float(row0["hotel_lat"]), float(row0["hotel_lon"])
print(row0["hotel_code"], row0["hotel_name"], lat, lon)

feats = ProximityFeatures().for_point(lat, lon)
series = pd.Series(feats).sort_index()

# Aperçu plage + agrégats F&B
preview_keys = (
    [f"plage_{k}km" for k in BEACH_RADII_KM]
    + ["plage_distance_km"]
    + [f"commerce_fb_{r}m" for r in COMMERCE_RADII_M]
    + [f"commerce_non_fb_{r}m" for r in COMMERCE_RADII_M]
)
display(series.loc[[k for k in preview_keys if k in series.index]])
print(f"… {len(series)} features au total")
series

H2075 Ibis budget Nice Californie 43.689186 7.240512


plage_1km               1.000000
plage_2km               1.000000
plage_3km               1.000000
plage_4km               1.000000
plage_5km               1.000000
plage_distance_km       0.127839
commerce_fb_100m        1.000000
commerce_fb_200m        2.000000
commerce_fb_300m        3.000000
commerce_fb_400m        7.000000
commerce_fb_500m        8.000000
commerce_non_fb_100m    0.000000
commerce_non_fb_200m    0.000000
commerce_non_fb_300m    0.000000
commerce_non_fb_400m    0.000000
commerce_non_fb_500m    0.000000
dtype: float64

… 91 features au total


commerce_alcohol_100m    0.000000
commerce_alcohol_200m    0.000000
commerce_alcohol_300m    0.000000
commerce_alcohol_400m    0.000000
commerce_alcohol_500m    0.000000
                           ...   
plage_2km                1.000000
plage_3km                1.000000
plage_4km                1.000000
plage_5km                1.000000
plage_distance_km        0.127839
Length: 91, dtype: float64

## 3. Pipeline complet → `Output/`

Une ligne par `hotel_code` Accor. Peut prendre 1–2 min (Overpass + pause anti rate-limit).

In [19]:
proximity = prep.run()
print("shape:", proximity.shape)
print("geo_source:", proximity["geo_source"].value_counts().to_dict())

show_cols = (
    ["hotel_code", "hotel_name", "geo_source", "plage_distance_km"]
    + [f"plage_{k}km" for k in BEACH_RADII_KM]
    + [f"commerce_fb_{r}m" for r in COMMERCE_RADII_M]
)
proximity[[c for c in show_cols if c in proximity.columns]]

shape: (7, 97)
geo_source: {'rod_coords': 7}


,hotel_code,hotel_name,geo_source,plage_distance_km,plage_1km,plage_2km,plage_3km,plage_4km,plage_5km,commerce_fb_100m,commerce_fb_200m,commerce_fb_300m,commerce_fb_400m,commerce_fb_500m
0,H2075,Ibis budget Nice Californie,rod_coords,0.127839,1.0,1.0,1.0,1.0,1.0,1.0,2.0,3.0,7.0,8.0
1,HB6A3,Ibis budget Strasbourg Centre République,rod_coords,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,H0815,Ibis Styles Roissy CDG,rod_coords,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0
3,H6188,Mercure Paris Boulogne,rod_coords,1.420828,0.0,1.0,1.0,1.0,1.0,0.0,3.0,4.0,5.0,16.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,rod_coords,3.265612,0.0,0.0,0.0,1.0,1.0,2.0,4.0,17.0,47.0,68.0
5,HB5I0,Novotel Megève Mont-Blanc,rod_coords,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,H3546,Novotel Paris Centre Tour Eiffel,rod_coords,2.104029,0.0,0.0,1.0,1.0,1.0,0.0,2.0,2.0,12.0,21.0


## 4. Contrôles rapides

In [20]:
assert proximity["hotel_code"].notna().all()
assert (proximity["hotel_code"] != proximity["hotel_name"]).all()
for r in COMMERCE_RADII_M:
    assert f"commerce_fb_{r}m" in proximity.columns
for k in BEACH_RADII_KM:
    assert f"plage_{k}km" in proximity.columns

print("Output :", OUTPUT_DIR / "proximity.parquet")
print("OK — schéma commerces 100–500 m + plage 1–5 km")

Output : /media/laghmari/ssd-data/dev/hotels/prepare/ProximityPrep/Output/proximity.parquet
OK — schéma commerces 100–500 m + plage 1–5 km
